In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import plotly.graph_objects as go
import gradio as gr

ARTIFACTS = Path("artifacts")
if not (ARTIFACTS / "hyperparams.json").exists():
    raise FileNotFoundError("artifacts/ is missing - run `uv run python main.py train` first.")

SPLITS = ["train", "val", "test"]

# Processed splits saved by the training stage
data = np.load(ARTIFACTS / "splits.npz")
X = {s: data[f"X_{s}"].astype(np.float32) for s in SPLITS}
Y = {s: data[f"y_{s}"].astype(np.float64) for s in SPLITS}

with open(ARTIFACTS / "hyperparams.json") as f:
    HP = json.load(f)
with open(ARTIFACTS / "baseline.json") as f:
    BASELINE_VALUE = json.load(f)["baseline_value"]
with open(ARTIFACTS / "feature_names.json") as f:
    FEATURE_NAMES = json.load(f)
results = pd.read_csv(ARTIFACTS / "results.csv")

# Original (un-imputed) data, used only for the distribution plots.
# Rows are in the same chronological order as train -> val -> test, so we label them by position.
raw = pd.read_csv(ARTIFACTS / "raw_view.csv", parse_dates=["dteday"])
assert len(raw) == sum(len(Y[s]) for s in SPLITS), "raw_view.csv doesn't match the saved splits"
raw["split"] = np.repeat(SPLITS, [len(Y[s]) for s in SPLITS])
print({s: len(Y[s]) for s in SPLITS})

In [ ]:
# Load the three TRAINED models from disk. Nothing is trained or fitted in this notebook.

class LinReg(nn.Module):
    """Same architecture as in train.ipynb - needed only so the saved weights can be loaded."""
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

sk_model = joblib.load(ARTIFACTS / "sklearn_ridge.joblib")
manual = torch.load(ARTIFACTS / "manual_torch.pt")
module = torch.load(ARTIFACTS / "module_torch.pt")

nn_model = LinReg(module["n_features"])
nn_model.load_state_dict(module["state_dict"])
nn_model.eval()

MODEL_NAMES = ["scikit-learn Ridge", "manual PyTorch", "nn.Module + optim"]
BASELINE_NAME = "baseline (train mean)"

def clip0(p):
    return np.clip(p, 0, None)          # rentals can't be negative (same rule as in training)

def predict_all(split):
    x = torch.tensor(X[split])
    with torch.no_grad():
        p_manual = (x @ manual["w"] + manual["b"]).squeeze(1).numpy() * manual["y_std"] + manual["y_mean"]
        p_module = nn_model(x).squeeze(1).numpy() * module["y_std"] + module["y_mean"]
    return {
        "scikit-learn Ridge": clip0(sk_model.predict(pd.DataFrame(X[split], columns=FEATURE_NAMES)).astype(np.float64)),
        "manual PyTorch": clip0(p_manual.astype(np.float64)),
        "nn.Module + optim": clip0(p_module.astype(np.float64)),
    }

# PRED[split][model_name] -> predictions in bikes/hour
PRED = {s: predict_all(s) for s in SPLITS}
for s in SPLITS:
    PRED[s][BASELINE_NAME] = np.full(len(Y[s]), BASELINE_VALUE)

# Cross-check: the reloaded models must reproduce the predictions saved during training
saved = np.load(ARTIFACTS / "predictions.npz")
KEY = {"scikit-learn Ridge": "sk", "manual PyTorch": "manual", "nn.Module + optim": "module"}
worst = max(np.abs(PRED[s][m] - saved[f"{KEY[m]}_{s}"]).max() for s in SPLITS for m in MODEL_NAMES)
CHECK_MSG = f"Reloaded models reproduce the saved training predictions (max difference {worst:.4f} bikes/hour)."
print(CHECK_MSG)

In [ ]:
COLORS = {
    "scikit-learn Ridge": "#1f77b4",
    "manual PyTorch": "#ff7f0e",
    "nn.Module + optim": "#2ca02c",
    BASELINE_NAME: "#7f7f7f",
}

# ---------- Tab 1: prediction vs actual ----------
def tab_compare(split, models):
    y = Y[split]
    fig = go.Figure()
    top = float(y.max())
    for m in models:
        p = PRED[split][m]
        top = max(top, float(p.max()))
        fig.add_trace(go.Scattergl(
            x=y, y=p, mode="markers", name=m,
            marker=dict(size=4, opacity=0.35, color=COLORS[m])))
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", name="perfect prediction",
                             line=dict(dash="dash", color="black")))
    fig.update_layout(title=f"Predicted vs actual hourly rentals ({split} set)",
                      xaxis_title="Actual rentals per hour",
                      yaxis_title="Predicted rentals per hour",
                      height=520, margin=dict(t=60, b=40))

    table = (results[results["split"] == split][["model", "RMSE", "MAE", "bias"]]
             .round(2).reset_index(drop=True))

    base = PRED[split]["scikit-learn Ridge"]
    diffs = {m: float(np.abs(PRED[split][m] - base).max()) for m in MODEL_NAMES[1:]}
    note = (f"**Agreement on the {split} set** - largest prediction gap vs scikit-learn: "
            + ", ".join(f"{m}: {d:.3f} bikes/hour" for m, d in diffs.items())
            + ". If the dots overlap almost perfectly, that is why: the three models make nearly identical predictions.")
    return fig, table, note


# ---------- Tab 2: distributions ----------
VAR_LABELS = {
    "cnt": "cnt - hourly rentals (target)",
    "temp": "temp (normalized)",
    "hum": "hum (humidity, before the sensor fix)",
    "windspeed": "windspeed (normalized)",
    "hr": "hr - hour of day",
}

def tab_distribution(var, splits):
    fig = go.Figure()
    for s in splits:
        vals = raw.loc[raw["split"] == s, var]
        fig.add_trace(go.Histogram(x=vals, name=s, opacity=0.55,
                                   histnorm="probability density", nbinsx=40))
    fig.update_layout(barmode="overlay", title=f"Distribution of {VAR_LABELS[var]}",
                      xaxis_title=VAR_LABELS[var], yaxis_title="density",
                      height=450, margin=dict(t=60, b=40))
    rows = []
    for s in splits:
        v = raw.loc[raw["split"] == s, var]
        rows.append({"split": s, "rows": len(v), "mean": v.mean(), "std": v.std(),
                     "min": v.min(), "median": v.median(), "max": v.max()})
    stats = pd.DataFrame(rows).round(3)
    return fig, stats


# ---------- Tab 3: business cost ----------
def short_and_surplus(y, p):
    short = np.maximum(y - p, 0)      # bikes we failed to provide (under-prediction)
    surplus = np.maximum(p - y, 0)    # idle bikes (over-prediction)
    return short, surplus

def tab_cost(split, k, buffer, model, daytype):
    y = Y[split]

    # 1) Cost per model (buffer is added to every model's prediction)
    rows = []
    for name in MODEL_NAMES + [BASELINE_NAME]:
        p = PRED[split][name] + buffer
        short, surplus = short_and_surplus(y, p)
        rows.append({
            "model": name,
            "mean cost / hour": round(float(np.mean(k * short + surplus)), 2),
            "avg bikes short / hour": round(float(short.mean()), 1),
            "avg idle bikes / hour": round(float(surplus.mean()), 1),
            "% hours under-predicted": round(float(np.mean(p < y) * 100), 1),
            "RMSE": round(float(np.sqrt(np.mean((y - p) ** 2))), 2),
        })
    table = pd.DataFrame(rows)

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(x=table["model"], y=[r["avg bikes short / hour"] * k for r in rows],
                          name=f"shortage cost (x{k:g})", marker_color="#d62728"))
    fig1.add_trace(go.Bar(x=table["model"], y=[r["avg idle bikes / hour"] for r in rows],
                          name="idle-bike cost", marker_color="#aec7e8"))
    fig1.update_layout(barmode="stack", title=f"Mean cost per hour ({split} set, buffer = {buffer:g} bikes)",
                       yaxis_title="cost units per hour", height=380, margin=dict(t=60, b=40))

    # 2) Cost by hour of day for the chosen model
    p = PRED[split][model] + buffer
    short, surplus = short_and_surplus(y, p)
    meta = raw[raw["split"] == split].reset_index(drop=True)
    df_h = pd.DataFrame({"hr": meta["hr"], "workingday": meta["workingday"], "actual": y,
                         "short_cost": k * short, "idle_cost": surplus})
    if daytype == "Working days":
        df_h = df_h[df_h["workingday"] == 1]
    elif daytype == "Non-working days":
        df_h = df_h[df_h["workingday"] == 0]
    g = df_h.groupby("hr").mean(numeric_only=True)

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=g.index, y=g["short_cost"], name="shortage cost", marker_color="#d62728"))
    fig2.add_trace(go.Bar(x=g.index, y=g["idle_cost"], name="idle-bike cost", marker_color="#aec7e8"))
    fig2.add_trace(go.Scatter(x=g.index, y=g["actual"], name="avg actual rentals (right axis)",
                              yaxis="y2", mode="lines", line=dict(color="black", dash="dot")))
    fig2.update_layout(barmode="stack",
                       title=f"Cost by hour of day - {model} ({daytype.lower()}, {split} set)",
                       xaxis_title="hour of day", yaxis_title="mean cost per hour",
                       yaxis2=dict(title="avg actual rentals", overlaying="y", side="right", showgrid=False),
                       height=450, margin=dict(t=60, b=40),
                       legend=dict(orientation="h", y=-0.2))
    return table, fig1, fig2

In [ ]:
with gr.Blocks(title="Bike-share demand dashboard") as demo:
    gr.Markdown(
        "# Capital Bikeshare - hourly demand dashboard\n"
        "Three implementations of the same Ridge regression, loaded from `artifacts/` "
        "(nothing is retrained here). Predictions are clipped at 0.\n\n"
        f"*{CHECK_MSG}*"
    )

    # ---- Tab 1 ----
    with gr.Tab("Model comparison"):
        with gr.Row():
            split_in = gr.Radio(SPLITS, value="test", label="Dataset split")
            models_in = gr.CheckboxGroup(MODEL_NAMES, value=MODEL_NAMES, label="Models to plot")
        cmp_plot = gr.Plot(label="Predicted vs actual")
        cmp_note = gr.Markdown()
        cmp_table = gr.Dataframe(label="Metrics from training (RMSE and MAE in bikes/hour; bias = mean(pred - actual))")

        cmp_inputs = [split_in, models_in]
        cmp_outputs = [cmp_plot, cmp_table, cmp_note]
        for c in cmp_inputs:
            c.change(tab_compare, cmp_inputs, cmp_outputs)

    # ---- Tab 2 ----
    with gr.Tab("Distributions"):
        with gr.Row():
            var_in = gr.Dropdown(choices=[(v, k_) for k_, v in VAR_LABELS.items()], value="cnt", label="Variable")
            dsplits_in = gr.CheckboxGroup(SPLITS, value=SPLITS, label="Splits to overlay")
        dist_plot = gr.Plot(label="Distribution")
        dist_stats = gr.Dataframe(label="Summary statistics")

        dist_inputs = [var_in, dsplits_in]
        dist_outputs = [dist_plot, dist_stats]
        for c in dist_inputs:
            c.change(tab_distribution, dist_inputs, dist_outputs)

    # ---- Tab 3 ----
    with gr.Tab("Business cost"):
        gr.Markdown(
            "**Framing:** an hour where we predict too few bikes means empty stations and lost rentals; "
            "predicting too many only leaves idle bikes. Each idle bike costs **1 unit**; each missing bike costs "
            "**k units**. The *safety buffer* adds the same number of bikes to every prediction - the regression "
            "version of moving a decision threshold. *(The value of k is an assumption - set it to what you justify in your report.)*"
        )
        with gr.Row():
            csplit_in = gr.Radio(["val", "test"], value="test", label="Dataset split")
            k_in = gr.Slider(1, 10, value=3, step=0.5, label="k = cost of a missing bike (idle bike = 1)")
            buf_in = gr.Slider(0, 150, value=0, step=5, label="Safety buffer (bikes added to every prediction)")
        with gr.Row():
            cmodel_in = gr.Dropdown(MODEL_NAMES, value="scikit-learn Ridge", label="Model for the by-hour view")
            day_in = gr.Radio(["All days", "Working days", "Non-working days"], value="Working days", label="Day type (by-hour view)")
        cost_table = gr.Dataframe(label="Cost per model")
        cost_plot1 = gr.Plot()
        cost_plot2 = gr.Plot()

        cost_inputs = [csplit_in, k_in, buf_in, cmodel_in, day_in]
        cost_outputs = [cost_table, cost_plot1, cost_plot2]
        for c in cost_inputs:
            c.change(tab_cost, cost_inputs, cost_outputs)

    # Fill every tab with its default view when the page opens
    demo.load(tab_compare, cmp_inputs, cmp_outputs)
    demo.load(tab_distribution, dist_inputs, dist_outputs)
    demo.load(tab_cost, cost_inputs, cost_outputs)

demo.launch(inbrowser=True)